# Квантизация весов FP16 → INT4 на Triton

Реализация первого пункта: квантизация по строкам, упаковка по два 4-битных значения в `int8`. 

**Формат весов для GEMM:**

- `W`: `[out_features, in_features]`, FP16 или BF16, `in_features` — чётное.
- `packed`: `[out_features, in_features // 2]`, `torch.int8`. Упаковка по последней размерности: `W[r, 2*j]` в младших 4 битах, `W[r, 2*j+1]` в старших.
- Signed INT4 в дополнительном коде, фактически используемый диапазон `[-7, 7]` (`-8` не используется).
- `scales`: `[out_features]`, FP32. Для строки `r`: `scale[r] = max(abs(W[r])) / 7`; для нулевой строки `scale = 1`. Zero-point = 0, отдельно не хранится.
- Восстановление: `W_hat[r, k] = signed_int4(packed[r, k//2], k%2) * scales[r]`.

In [1]:
import torch
import triton
import triton.language as tl
import triton.testing
import statistics
import json

assert torch.cuda.is_available(), "В Colab нужно включить GPU"
torch.manual_seed(42)
DEVICE = "cuda"
print(torch.cuda.get_device_name(0), '| PyTorch', torch.__version__, '| Triton', triton.__version__)

Tesla T4 | PyTorch 2.14.1+cu130 | Triton 3.8.0


## Кернели

In [2]:
@triton.jit
def row_max_kernel(X, RowMax, N: tl.constexpr, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    row_max = tl.full((), 0.0, tl.float32)
    for start in range(tl.cdiv(N, BLOCK_N)):
        j = start * BLOCK_N + cols
        x = tl.load(X + row * N + j, mask=j < N, other=0.0).to(tl.float32)
        row_max = tl.maximum(row_max, tl.max(tl.abs(x), 0))
    tl.store(RowMax + row, row_max)


@triton.jit
def quantize_pack_kernel(X, RowMax, Packed, Scales,
                         N: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    block = tl.program_id(1)
    max_val = tl.load(RowMax + row)

    scale = tl.where(max_val > 0.0, tl.div_rn(max_val, 7.0), 1.0)
    if block == 0:
        tl.store(Scales + row, scale)

    j = block * BLOCK + tl.arange(0, BLOCK)
    even = 2 * j
    odd = even + 1
    x0 = tl.load(X + row * N + even, mask=even < N, other=0.0).to(tl.float32)
    x1 = tl.load(X + row * N + odd, mask=odd < N, other=0.0).to(tl.float32)

    a0 = tl.abs(x0) * 14.0
    a1 = tl.abs(x1) * 14.0
    q0_abs = tl.full((BLOCK,), 0, tl.int32)
    q1_abs = tl.full((BLOCK,), 0, tl.int32)
    for i in tl.static_range(0, 7):
        threshold = max_val * (2 * i + 1)
        q0_abs += ((max_val > 0.0) & (a0 >= threshold)).to(tl.int32)
        q1_abs += ((max_val > 0.0) & (a1 >= threshold)).to(tl.int32)

    q0 = tl.where(x0 < 0.0, -q0_abs, q0_abs)
    q1 = tl.where(x1 < 0.0, -q1_abs, q1_abs)
    byte = ((q0 & 15) | ((q1 & 15) << 4)).to(tl.int8)
    tl.store(Packed + row * (N // 2) + j, byte, mask=j < N // 2)

### Альтернатива: fused row-wise quantize + pack

Один Triton program обрабатывает одну строку целиком: один проход чтения весов, `absmax`, scale, округление и упаковка. Выход **побайтово совместим** с двухпроходным вариантом (та же пороговая арифметика). Для больших `N` может быть медленнее из-за давления на регистры; бенчмарк ниже проверяет это экспериментально.

In [3]:
@triton.jit
def fused_quantize_pack_kernel(X, Packed, Scales,
                               N: tl.constexpr, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    j = tl.arange(0, BLOCK_N)
    x = tl.load(X + row * N + j, mask=j < N, other=0.0).to(tl.float32)
    max_val = tl.max(tl.abs(x), 0)
    scale = tl.where(max_val > 0.0, tl.div_rn(max_val, 7.0), 1.0)
    tl.store(Scales + row, scale)

    # Квантизация выполняется над всеми элементами строки (пороговая
    # арифметика как в baseline), затем упаковываются пары соседей.
    a = tl.abs(x) * 14.0
    q_abs = tl.full((BLOCK_N,), 0, tl.int32)
    for k in tl.static_range(0, 7):
        threshold = max_val * (2 * k + 1)
        q_abs += ((max_val > 0.0) & (a >= threshold)).to(tl.int32)
    q = tl.where(x < 0.0, -q_abs, q_abs)

    # Gather по четным и нечетным позициям; по одной паре на байт.
    pairs = tl.arange(0, BLOCK_N // 2)
    q0 = tl.gather(q, 2 * pairs, axis=0)
    q1 = tl.gather(q, 2 * pairs + 1, axis=0)
    packed = ((q0 & 15) | ((q1 & 15) << 4)).to(tl.int8)
    tl.store(Packed + row * (N // 2) + pairs, packed, mask=pairs < N // 2)

## Квантизация и проверка распаковки

In [4]:
def quantize_fp16_to_int4_packed(W, method="two_pass"):
    """Симметричная per-row FP16/BF16 -> packed INT4 + FP32 scales.

    method: "two_pass" (baseline) или "fused"; выходной формат одинаков.
    Контракт входа: CUDA, конечные числа, 2D, N чётное.
    Проверка конечности не выполняется внутри, чтобы не добавлять GPU sync.
    """
    assert W.is_cuda and W.ndim == 2
    assert W.dtype in (torch.float16, torch.bfloat16)
    M, N = W.shape
    assert M > 0 and N > 0 and N % 2 == 0
    W = W.contiguous()
    if method not in ("two_pass", "fused"):
        raise ValueError("method must be 'two_pass' or 'fused'")
    packed = torch.empty((M, N // 2), device=W.device, dtype=torch.int8)
    scales = torch.empty((M,), device=W.device, dtype=torch.float32)
    if method == "two_pass":
        row_max = torch.empty((M,), device=W.device, dtype=torch.float32)
        row_max_kernel[(M,)](W, row_max, N, BLOCK_N=1024, num_warps=4)
        B = 256
        quantize_pack_kernel[(M, triton.cdiv(N // 2, B))](
            W, row_max, packed, scales, N, B, num_warps=4
        )
    else:
        fused_quantize_pack_kernel[(M,)](
            W, packed, scales, N, BLOCK_N=triton.next_power_of_2(N), num_warps=4
        )
    return packed, scales


def decode_signed_int4(packed):
    """Возвращает signed целые [-7,7] в форме [M,N]."""
    p = packed.to(torch.int32)
    lo = p & 15
    hi = (p >> 4) & 15
    lo = torch.where(lo >= 8, lo - 16, lo)
    hi = torch.where(hi >= 8, hi - 16, hi)
    M, half = packed.shape
    q = torch.empty((M, 2 * half), device=packed.device, dtype=torch.int32)
    q[:, 0::2], q[:, 1::2] = lo, hi
    return q


def dequantize_int4_packed(packed, scales):
    return decode_signed_int4(packed).float() * scales[:, None]


def pytorch_reference(W):
    wf = W.float()
    row_max = wf.abs().amax(dim=1)
    scales = torch.where(row_max > 0, row_max / 7.0, torch.ones_like(row_max))
    abs14 = wf.abs() * 14.0
    qabs = torch.zeros_like(wf, dtype=torch.int32)
    for i in range(7):
        t = row_max[:, None] * (2 * i + 1)
        qabs += ((row_max[:, None] > 0) & (abs14 >= t)).int()
    q = torch.where(wf < 0, -qabs, qabs)
    packed = ((q[:, 0::2] & 15) | ((q[:, 1::2] & 15) << 4)).to(torch.int8)
    return packed, scales


def memory_bytes(*tensors):
    return sum(x.numel() * x.element_size() for x in tensors)

def independent_round_reference(W):
    """Математически независимый reference: half-away-from-zero, FP32.

    Из-за отличий FP32 операций от порогового метода возможны расхождения
    ровно у границ округления; их число диагностируем, но не скрываем.
    """
    wf = W.float()
    row_max = wf.abs().amax(dim=1, keepdim=True)
    scale = torch.where(row_max > 0, row_max / 7.0, torch.ones_like(row_max))
    qabs = torch.floor(wf.abs() / scale + 0.5).clamp_(0, 7).to(torch.int32)
    q = torch.where(wf < 0, -qabs, qabs)
    return q, scale.squeeze(1)


In [5]:
def check_case(W, label, method="two_pass"):
    assert torch.isfinite(W).all().item(), "Тестовые веса должны быть конечными"
    M, N = W.shape
    packed, scales = quantize_fp16_to_int4_packed(W, method=method)
    ref_packed, ref_scales = pytorch_reference(W)
    decoded = decode_signed_int4(packed)
    restored = decoded.float() * scales[:, None]
    independent_q, _ = independent_round_reference(W)
    independent_diff = (decoded != independent_q).sum().item()
    assert packed.shape == (M, N // 2) and packed.dtype == torch.int8
    assert scales.shape == (M,) and scales.dtype == torch.float32
    assert decoded.min().item() >= -7 and decoded.max().item() <= 7
    assert torch.isfinite(restored).all().item()

    mismatches = (packed != ref_packed).sum().item()
    scale_delta = (scales - ref_scales).abs().max().item()
    scales_ok = torch.allclose(scales, ref_scales, rtol=1e-6, atol=1e-7)
    err = (W.float() - restored).abs()
    # Возможно микроскопическое расхождение FP32 scale при деквантизации.
    bound_violations = (err > 0.5 * scales[:, None] + 1e-5).sum().item()
    before = memory_bytes(W)
    after = memory_bytes(packed, scales)
    ratio = before / after
    mse = err.square().mean().item()
    print(f"{label:22s} {M:5d}x{N:<5d} {str(W.dtype):14s} "
          f"mismatch={mismatches:5d} scale_diff={scale_delta:.2e} "
          f"MAE={err.mean().item():.6f} max={err.max().item():.6f} "
          f"MSE={mse:.6g} independent_diff={independent_diff} "
          f"bound_fail={bound_violations} compression={ratio:.4f}x")
    if mismatches:
        for r, c in (packed != ref_packed).nonzero()[:5].tolist():
            print(f"  byte [{r},{c}]: Triton={int(packed[r,c]) & 255:02x}, "
                  f"PyTorch={int(ref_packed[r,c]) & 255:02x}")
    assert scales_ok, "Scales diverged"
    assert mismatches == 0, "Packed bytes differ from PyTorch reference"
    assert bound_violations == 0, "Quantization error bound failed"
    assert memory_bytes(packed) * 4 == before, "Packed bytes not exactly quarter FP16/BF16"
    return {"label": label, "shape": [M, N], "dtype": str(W.dtype),
            "mismatches": mismatches, "mae": err.mean().item(),
            "max_error": err.max().item(), "mse": mse,
            "independent_q_mismatches": independent_diff,
            "compression": ratio, "method": method}

In [6]:
print("Проверка корректности")
test_results = []
for M, N in [(1, 128), (13, 256), (2048, 2048), (8192, 2048)]:
    test_results.append(check_case(torch.randn(M, N, device=DEVICE, dtype=torch.float16), "random FP16"))
test_results.append(check_case(torch.randn(32, 256, device=DEVICE, dtype=torch.bfloat16), "random BF16"))
test_results.append(check_case(torch.zeros(3, 128, device=DEVICE, dtype=torch.float16), "all zero"))

mixed = torch.randn(5, 256, device=DEVICE, dtype=torch.float16)
mixed[0] = 0
mixed[3] = 0
test_results.append(check_case(mixed, "mixed zero rows"))

Проверка корректности
random FP16                1x128   torch.float16  mismatch=    0 scale_diff=2.98e-08 MAE=0.089752 max=0.177734 MSE=0.0108199 independent_diff=0 bound_fail=0 compression=3.7647x
random FP16               13x256   torch.float16  mismatch=    0 scale_diff=5.96e-08 MAE=0.104574 max=0.261509 MSE=0.0150031 independent_diff=2 bound_fail=0 compression=3.8788x
random FP16             2048x2048  torch.float16  mismatch=    0 scale_diff=5.96e-08 MAE=0.129597 max=0.380301 MSE=0.022577 independent_diff=672 bound_fail=0 compression=3.9844x
random FP16             8192x2048  torch.float16  mismatch=    0 scale_diff=5.96e-08 MAE=0.129753 max=0.384347 MSE=0.0226299 independent_diff=2480 bound_fail=0 compression=3.9844x
random BF16               32x256   torch.bfloat16 mismatch=    0 scale_diff=5.96e-08 MAE=0.112683 max=0.300502 MSE=0.0171546 independent_diff=20 bound_fail=0 compression=3.8788x
all zero                   3x128   torch.float16  mismatch=    0 scale_diff=0.00e+00 MAE

In [7]:
# Проверка знаков, точных порогов и маленькой матрицы
small = torch.tensor([[1,-2,3,-4,5,-6,7,-8],
                      [.5,.5,.5,.5,-.5,-.5,-.5,-.5],
                      [2,2,2,2,2,2,2,2],
                      [0,0,0,0,0,0,0,0]], device=DEVICE, dtype=torch.float16)
test_results.append(check_case(small, "small example"))

# Включая границы порогов 0.5,1.5,...,6.5 для max=7
boundaries = torch.tensor([[7., .5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5,
                            -7., -.5, -1.5, -2.5, -3.5, -4.5, -5.5, -6.5]],
                          device=DEVICE, dtype=torch.float16)
test_results.append(check_case(boundaries, "threshold ties"))

zero_p, zero_s = quantize_fp16_to_int4_packed(torch.zeros(3,128,device=DEVICE,dtype=torch.float16))
assert torch.count_nonzero(zero_p).item() == 0
assert torch.all(zero_s == 1).item()

for bad_n in [3, 7]:
    try:
        quantize_fp16_to_int4_packed(torch.zeros((2,bad_n),device=DEVICE,dtype=torch.float16))
        raise RuntimeError("Odd-width input was unexpectedly accepted")
    except AssertionError:
        pass

small example              4x8     torch.float16  mismatch=    0 scale_diff=0.00e+00 MAE=0.071429 max=0.571429 MSE=0.0280612 independent_diff=1 bound_fail=0 compression=2.0000x
threshold ties             1x16    torch.float16  mismatch=    0 scale_diff=0.00e+00 MAE=0.437500 max=0.500000 MSE=0.21875 independent_diff=0 bound_fail=0 compression=2.6667x


In [8]:
# Дополнительные контрактные проверки: положение тетрад, знаковость,
# 0-строки и не-contiguous вход.
patterns = torch.tensor([[7., -7., -1., 1., 0., 0., 3., -3.]],
                        device=DEVICE, dtype=torch.float16)
pat_p, pat_s = quantize_fp16_to_int4_packed(patterns)
assert (pat_p.to(torch.int32) & 255).tolist() == [[0x97, 0x1f, 0x00, 0xd3]]
assert decode_signed_int4(pat_p).tolist() == [[7,-7,-1,1,0,0,3,-3]]

base = torch.randn(256, 16, device=DEVICE, dtype=torch.float16)
noncontig = base[:, ::2]  # K=8, stride K=2
assert not noncontig.is_contiguous()
p_nc, s_nc = quantize_fp16_to_int4_packed(noncontig)
r_nc, rs_nc = pytorch_reference(noncontig)
assert torch.equal(p_nc, r_nc)
assert torch.allclose(s_nc, rs_nc, atol=1e-7, rtol=1e-6)

print("Additional contract tests: PASS")

Additional contract tests: PASS


### Расширенные проверки: края, outliers и сравнение двух реализаций

Дополнительные тесты включают неполные блоки, минимальные ширины, BF16, subnormal FP16, выбросы и идентичность двух методов. Второй, математически независимый reference выводит число расхождений округления (они могут возникать на границах FP32), а не требует побайтового совпадения.

In [9]:
for M, N in [(1, 2), (3, 6), (7, 130), (9, 514), (4, 2048)]:
    for dtype in (torch.float16, torch.bfloat16):
        W = torch.randn((M, N), device=DEVICE, dtype=dtype)
        test_results.append(check_case(W, f"edge {dtype}", method="two_pass"))
        p_base, s_base = quantize_fp16_to_int4_packed(W, method="two_pass")
        p_fused, s_fused = quantize_fp16_to_int4_packed(W, method="fused")
        assert torch.equal(p_base, p_fused), f"Packed mismatch on {M}x{N} {dtype}"
        assert torch.equal(s_base, s_fused), f"Scales mismatch on {M}x{N} {dtype}"

special = torch.tensor([[0, 2**-24, -2**-24, 2**-14, 0.0, 0.0],
                        [0.01, -0.02, 0.03, 0.04, 0.05, 60.0]],
                       device=DEVICE, dtype=torch.float16)
test_results.append(check_case(special, "subnormal/outlier"))
p0, s0 = quantize_fp16_to_int4_packed(special)
p1, s1 = quantize_fp16_to_int4_packed(special, method="fused")
assert torch.equal(p0, p1) and torch.equal(s0, s1)
print("Extended and fused equivalence tests: PASS")

edge torch.float16         1x2     torch.float16  mismatch=    0 scale_diff=1.49e-08 MAE=0.048096 max=0.096191 MSE=0.00462639 independent_diff=0 bound_fail=0 compression=0.8000x
edge torch.bfloat16        1x2     torch.bfloat16 mismatch=    0 scale_diff=1.49e-08 MAE=0.001883 max=0.003767 MSE=7.09419e-06 independent_diff=0 bound_fail=0 compression=0.8000x
edge torch.float16         3x6     torch.float16  mismatch=    0 scale_diff=2.98e-08 MAE=0.065423 max=0.151437 MSE=0.00682141 independent_diff=0 bound_fail=0 compression=1.7143x
edge torch.bfloat16        3x6     torch.bfloat16 mismatch=    0 scale_diff=2.98e-08 MAE=0.063918 max=0.130580 MSE=0.00587251 independent_diff=0 bound_fail=0 compression=1.7143x
edge torch.float16         7x130   torch.float16  mismatch=    0 scale_diff=2.98e-08 MAE=0.099330 max=0.226562 MSE=0.0134155 independent_diff=0 bound_fail=0 compression=3.7681x
edge torch.bfloat16        7x130   torch.bfloat16 mismatch=    0 scale_diff=2.98e-08 MAE=0.096167 max=0.242188

## Пример

In [10]:
p, s = quantize_fp16_to_int4_packed(small)
print("Веса:\n", small.cpu().numpy())
print("Packed int8:\n", p.cpu().numpy())
print("Scales:\n", s.cpu().numpy())
print("После деквантизации:\n", dequantize_int4_packed(p, s).cpu().numpy())

Веса:
 [[ 1.  -2.   3.  -4.   5.  -6.   7.  -8. ]
 [ 0.5  0.5  0.5  0.5 -0.5 -0.5 -0.5 -0.5]
 [ 2.   2.   2.   2.   2.   2.   2.   2. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0. ]]
Packed int8:
 [[ -31  -61  -76 -106]
 [ 119  119 -103 -103]
 [ 119  119  119  119]
 [   0    0    0    0]]
Scales:
 [1.1428572  0.07142857 0.2857143  1.        ]
После деквантизации:
 [[ 1.1428572 -2.2857144  3.4285717 -4.571429   4.571429  -5.714286
   6.8571434 -8.       ]
 [ 0.5        0.5        0.5        0.5       -0.5       -0.5
  -0.5       -0.5      ]
 [ 2.         2.         2.         2.         2.         2.
   2.         2.       ]
 [ 0.         0.         0.         0.         0.         0.
   0.         0.       ]]


### Учёт памяти (включая scales)

`packed` содержит ровно четверть числа байт FP16/BF16. Однако реальные затраты — `M*N/2 + 4*M` байт вместе с FP32 `scales`. Поэтому реальное сжатие равно `4*N/(N+8)`, чуть меньше 4×, и зависит от ширины строки.

In [11]:
for M, N in [(2048, 2048), (8192, 2048), (8192, 8192)]:
    input_bytes = 2 * M * N
    packed_bytes = M * N // 2
    scales_bytes = 4 * M
    print(f"{M}x{N}: FP16={input_bytes/2**20:.2f} MiB, "
          f"packed={packed_bytes/2**20:.2f} MiB, "
          f"scales={scales_bytes/2**20:.4f} MiB, "
          f"total compression={input_bytes/(packed_bytes+scales_bytes):.4f}x")

2048x2048: FP16=8.00 MiB, packed=2.00 MiB, scales=0.0078 MiB, total compression=3.9844x
8192x2048: FP16=32.00 MiB, packed=8.00 MiB, scales=0.0312 MiB, total compression=3.9844x
8192x8192: FP16=128.00 MiB, packed=32.00 MiB, scales=0.0312 MiB, total compression=3.9961x


## Скорость квантизации (измеряется только подготовка весов)

In [ ]:
print("Бенчмарк: медиана 5 запусков, предвыделенные выходы (мс)")
bench_results = []
for M, N in [(2048, 2048), (8192, 2048), (8192, 8192)]:
    W = torch.randn((M, N), device=DEVICE, dtype=torch.float16)
    row_max = torch.empty((M,), device=DEVICE, dtype=torch.float32)
    packed = torch.empty((M, N//2), device=DEVICE, dtype=torch.int8)
    scales = torch.empty((M,), device=DEVICE, dtype=torch.float32)
    B = 256

    def run_two_pass():
        row_max_kernel[(M,)](W, row_max, N, BLOCK_N=1024, num_warps=4)
        quantize_pack_kernel[(M, triton.cdiv(N//2, B))](
            W, row_max, packed, scales, N, B, num_warps=4)

    def run_fused():
        fused_quantize_pack_kernel[(M,)](
            W, packed, scales, N, BLOCK_N=triton.next_power_of_2(N), num_warps=4)

    measurements = {}
    for name, fn in (("two_pass", run_two_pass), ("fused", run_fused)):
        fn()
        torch.cuda.synchronize()
        samples = [triton.testing.do_bench(fn, warmup=50, rep=150) for _ in range(5)]
        measurements[name] = {
            "median_ms": round(statistics.median(samples), 5),
            "samples_ms": [round(v, 5) for v in samples],
        }
    measurements["speedup_fused_vs_two_pass"] = round(
        measurements["two_pass"]["median_ms"] / measurements["fused"]["median_ms"], 4)
    bench_results.append({"shape": [M, N], **measurements})
    print(f"{M}x{N}: two_pass={measurements['two_pass']['median_ms']:.3f} мс, "
          f"fused={measurements['fused']['median_ms']:.3f} мс, "
          f"speedup={measurements['speedup_fused_vs_two_pass']:.2f}x")
    del W, row_max, packed, scales

Бенчмарк: медиана 5 запусков, предвыделенные выходы (мс)
2048x2048: two_pass=0.092 мс, fused=0.097 мс, speedup=0.95x
8192x2048: two_pass=0.333 мс, fused=0.380 мс, speedup=0.88x
8192x8192: two_pass=1.377 мс, fused=2.494 мс, speedup=0.55x


## Результаты и формат для команды

In [13]:
contract = {
    "packed_dtype": "torch.int8",
    "packed_shape": "[out_features, in_features // 2]",
    "pack_axis": "in_features",
    "low_nibble": "weight[2*j]",
    "high_nibble": "weight[2*j+1]",
    "int4": "signed two's complement, used values [-7, 7]",
    "scales_dtype": "torch.float32",
    "scales_shape": "[out_features]",
    "quantization": "symmetric per-row; scale = row_absmax/7 (or 1 for zero row)",
    "zero_point": 0,
    "methods": ["two_pass", "fused"],
    "input_precondition": "finite CUDA FP16/BF16, 2D, even in_features",
    "dequantization": "weight_hat[row, k] = signed_int4 * scales[row]",
}
results = {
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "triton": triton.__version__,
    "correctness": "PASS (if all notebook assertions completed)",
    "tests": test_results,
    "benchmarks": bench_results,
    "format": contract,
}
with open("int4_task1_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Результаты сохранены в int4_task1_results.json")
print("Интерфейс: packed, scales = quantize_fp16_to_int4_packed(W, method='two_pass' или 'fused')")

Результаты сохранены в int4_task1_results.json
Интерфейс: packed, scales = quantize_fp16_to_int4_packed(W, method='two_pass' или 'fused')
